parent: Preselection_v6

In [1]:
import os, sys
import numpy as np
import pickle
import uproot
import glob
import pandas as pd
from tqdm import tqdm
import json
import ROOT as rt
import fnmatch

## user define
sys.path.append('.')
sys.path.append('./Utils')
from Plotter import HistStack

# from beeps import *

Welcome to JupyROOT 6.28/06


# Set config

In [2]:
# ERA="ee17"
# ERA="ee16pre"
# # ERA="mm18"
# ERA="em18"
# # DATE = '20230129_17_eleffmva'
# DATE = '20230301_18'
# # DATE = '20230130_18_dyhighworctight'
# # INPUTDIR = "/eos/user/h/hkwon/Run2/20230130_18_dyhighworctight/mm/"
# INPUTDIR = "/eos/user/h/hkwon/Run2/20230301_18/em/"
import sys
ERA = sys.argv[1]
DATE = sys.argv[2]
INPUTDIR = sys.argv[3]

DUMP_PKL = False
DUMP_SIDEBAND_MET_PKL = False
DUMP_SIDEBAND_MASS_PKL = False
DUMP_SIDEBAND_PT_PKL = False
DUMP_DIVIDED_BY_BTAG = True
# DRAW_SR = True
# RUN_SIDEBAND_MET = False
# RUN_SIDEBAND_MASS = False
# RUN_SIDEBAND_PT = False
# RUN_BTAG = True
# EXPORT_VARIABLE = False

# for dev
# ERA = "ee18"

year = ERA.replace("em", "")
year = year.replace("mm", "")
year = year.replace("ee", "")
channel = ERA.replace("16pre", "")
channel = channel.replace("16post", "")
channel = channel.replace("17", "")
channel = channel.replace("18", "")

# DATE = "20231214GEv1"
# INPUTDIR = "/eos/user/h/hkwon/Run2/"+DATE+"_"+year+"/"+channel+"/"


# Get input

In [3]:
from warnings import simplefilter 
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

lumi_all={
    "ee16pre":19520,
    "ee16post":16810,
    "ee17":41480,
    "ee18":59830,
    "mm16pre":19520,
    "mm16post":16810,
    "mm17":41480,
    "mm18":59830,    
#     "mm18":14000,    
    "em16pre":19520,
    "em16post":16810,
    "em17":41480,
    "em18":59830     
}

data_evts = {
    "ee16pre":652566543,
    "ee16post":287977113,
    "ee17":460041183,
    "ee18":1326465857,
    "mm16pre":458655573,
    "mm16post":327037312,
    "mm17":739073837,
#     "mm18":944842550,  #949352777 
    "mm18":949352777,  #949352777 
    "em16pre":1111222116,
    "em16post":615014425,
    "em17":1199115020,
#     "em18":2271308407   #2281001366  
    "em18":2281001366   #2281001366  
}

lumi=lumi_all[ERA]
print(ERA, lumi)


def get_df(root_file_name, unneccesary_columns):
    f = uproot.open(root_file_name)
    df = uproot.open(root_file_name)["Events_"+channel].arrays(library="pd")
    df_met_u = uproot.open(root_file_name)["Events_"+channel+"_met_u"].arrays(library="pd")
    df_met_d = uproot.open(root_file_name)["Events_"+channel+"_met_d"].arrays(library="pd")     
    df_met_jer_u = uproot.open(root_file_name)["Events_"+channel+"_met_jer_u"].arrays(library="pd")
    df_met_jer_d = uproot.open(root_file_name)["Events_"+channel+"_met_jer_d"].arrays(library="pd")    
    df_met_ue_u = uproot.open(root_file_name)["Events_"+channel+"_met_ue_u"].arrays(library="pd")
    df_met_ue_d = uproot.open(root_file_name)["Events_"+channel+"_met_ue_d"].arrays(library="pd") 
    df_scale_u = uproot.open(root_file_name)["Events_"+channel+"_scale_u"].arrays(library="pd")
    df_scale_d = uproot.open(root_file_name)["Events_"+channel+"_scale_d"].arrays(library="pd")
    info = uproot.open(root_file_name)["Info_"+channel].arrays(library="pd")
#     return sum(info["sumweight"]), df.drop(unneccesary_columns, axis=1), df_met_u.drop(unneccesary_columns, axis=1), df_met_d.drop(unneccesary_columns, axis=1), df_scale_u.drop(unneccesary_columns, axis=1), df_scale_d.drop(unneccesary_columns, axis=1)
    return sum(info["sumweight"]), df.drop(unneccesary_columns, axis=1), df_met_u.drop(unneccesary_columns, axis=1), df_met_d.drop(unneccesary_columns, axis=1), df_met_jer_u.drop(unneccesary_columns, axis=1), df_met_jer_d.drop(unneccesary_columns, axis=1), df_met_ue_u.drop(unneccesary_columns, axis=1), df_met_ue_d.drop(unneccesary_columns, axis=1), df_scale_u.drop(unneccesary_columns, axis=1), df_scale_d.drop(unneccesary_columns, axis=1)

unneccesary_columns = []
dic_input = {
    "ee16pre": "/eos/user/h/hkwon/Run2/"+DATE+"_16pre/ee/",
    "mm16pre": "/eos/user/h/hkwon/Run2/"+DATE+"_16pre/mm/",
    "em16pre": "/eos/user/h/hkwon/Run2/"+DATE+"_16pre/em/",
    "ee16post": "/eos/user/h/hkwon/Run2/"+DATE+"_16post/ee/",
    "mm16post": "/eos/user/h/hkwon/Run2/"+DATE+"_16post/mm/",
    "em16post": "/eos/user/h/hkwon/Run2/"+DATE+"_16post/em/",
    "ee17": "/eos/user/h/hkwon/Run2/"+DATE+"_17/ee/",
    "mm17": "/eos/user/h/hkwon/Run2/"+DATE+"_17/mm/",
    "em17": "/eos/user/h/hkwon/Run2/"+DATE+"_17/em/",
    "ee18": "/eos/user/h/hkwon/Run2/"+DATE+"_18/ee/",
    "mm18": "/eos/user/h/hkwon/Run2/"+DATE+"_18/mm/",    
    "em18": "/eos/user/h/hkwon/Run2/"+DATE+"_18/em/"        
#     "em18": "/eos/user/h/hkwon/Run2/20221120_18/em/"        
}
# input_dir = dic_input[ERA]
input_dir = INPUTDIR
f = []
for (dirpath, dirnames, filenames) in os.walk(input_dir):
    f.extend(filenames)
    break

dic_tup={}
for file in tqdm(f):
    # file naming: flatNtuple_Preselect_WZ1.root, flatNtuple_Preselect_Zp-3700_CH-1345.root
    if not "flatNtuple" in file:
        continue
#     if channel=="mm" and (fnmatch.fnmatch(file, '*E*Run*') or fnmatch.fnmatch(file, '*M*RunB*') or fnmatch.fnmatch(file, '*M*RunC*') or fnmatch.fnmatch(file, '*M*RunD*')): #temp
#     if channel=="mm" and fnmatch.fnmatch(file, '*E*Run*'): #temp
#         continue
#     if channel=="ee" and fnmatch.fnmatch(file, '*M*Run*'):
#         continue
#     if channel=="em" and fnmatch.fnmatch(file, '*E*Run*'):
#         continue 
    if "Run" in file and not "hadd" in file:
        continue
    if "TT" in file and not "hadd" in file:
        continue
    if "DY" in file and not "hadd" in file and not "To" in file:
        continue  
    if "TW" in file and not "hadd" in file:
        continue         
    if "WW" in file and not "hadd" in file:
        continue  
    if "WZ" in file and not "hadd" in file:
        continue      
    if "ZZ" in file and not "hadd" in file:
        continue      
    if "Htt" in file and not "hadd" in file:
        continue  
        
    if "Zp" in file:
        sample=file.split('_')[2]+"_"+file.split('_')[3]
    else:
        print(file)
        sample=file.split('_')[2]
    sample=sample.rstrip('.root')
    try:
        print(sample)
        dic_tup[sample]=get_df(input_dir+file, unneccesary_columns)   
    except:
        print(sample)
        # beep()        
        if "Run" in sample: # check file corrupt
            raise    
# dic_tup

ee18 59830


  0%|          | 0/320 [00:00<?, ?it/s]

flatNtuple_Preselect_DY1000To15000.root
DY1000To15000


  1%|▏         | 4/320 [00:01<01:57,  2.69it/s]

flatNtuple_Preselect_DY1500To20000.root
DY1500To20000


...

 99%|█████████▉| 318/320 [02:29<00:04,  2.03s/it]

flatNtuple_Preselect_antitop_hadd.root
antitop


100%|█████████▉| 319/320 [02:31<00:01,  1.98s/it]

flatNtuple_Preselect_top_hadd.root
top


100%|██████████| 320/320 [02:33<00:00,  2.08it/s]


In [4]:
# beep_ok()

In [5]:
# for key in dic_tup:
#     print(key)

In [6]:
# get x-sec
dic_xsec={}
with open('config/CrossSection.json') as json_file:
    data = json.load(json_file)
    dic_xsec=data
print(dic_xsec)

{'Data': 1, 'DY': 6077.22, 'DY500To700': 0.2558, 'DY700To800': 0.04023, 'DY800To1000': 0.03406, 'DY1000To1500': 0.01828, 'DY1500To2000': 0.002367, 'DY2000To3000': 0.0005409, 'TT': 88.29, 'TTUp': 88.29, 'TTDown': 88.29, 'TThdampUp': 88.29, 'TThdampDown': 88.29, 'TTSemi': 365.34, 'TTZ': 0.2432, 'top': 34.91, 'antitop': 34.91, 'WW': 12.82, 'WZ': 5.23, 'WZQQ': 6.42, 'ZZ': 0.6008, 'WWW': 0.0072, 'WWZ': 0.1707, 'HWW': 0.31, 'H': 2.78, 'Zp-1700_CH-345': 0.03523, 'Zp-1700_CH-595': 0.03523, 'Zp-1700_CH-845': 0.03523, 'Zp-2100_CH-345': 0.01198, 'Zp-2100_CH-595': 0.01198, 'Zp-2100_CH-845': 0.01198, 'Zp-2500_CH-345': 0.004396, 'Zp-2500_CH-595': 0.004396, 'Zp-2500_CH-845': 0.004396, 'Zp-2500_CH-1095': 0.004396, 'Zp-2900_CH-345': 0.001768, 'Zp-2900_CH-595': 0.001768, 'Zp-2900_CH-845': 0.001768, 'Zp-2900_CH-1095': 0.001768, 'Zp-2900_CH-1345': 0.001768, 'Zp-3300_CH-345': 0.000729, 'Zp-3300_CH-595': 0.000729, 'Zp-3300_CH-845': 0.000729, 'Zp-3300_CH-1095': 0.000729, 'Zp-3300_CH-1345': 0.000729, 'Zp-3300

In [7]:
merged_dic_df={}
for key in dic_tup:       
    merged_dic_df[key]=dic_tup[key]

In [9]:
# # free memory 
# dic_tup.clear()

In [10]:
totalinput = """
######################################################
####   Total input number of events (sumweight)   ####
######################################################
"""


# merged_dic_df["Zp-2500"]=dic_tup["Zp-2500"][0], dic_tup["Zp-2500"][1] 
hadded = ["Data", "TT", "TTSemi", "TTUp", "TTDown", "TThdampUp", "TThdampDown", "top", "antitop", "WW", "WZ", "WZQQ", "DY", "TTZ", "ZZ", "H", "HWW", "WWW", "WWZ"]
for p in hadded:
    line = p + ":" + str(merged_dic_df[p][0]) + '\n'
    totalinput += line
    
print(totalinput)


######################################################
####   Total input number of events (sumweight)   ####
######################################################
Data:1326465857
TT:143848848
TTSemi:472557630
TTUp:56603364
TTDown:59369412
TThdampUp:53803428
TThdampDown:58968886
top:7250642
antitop:7748690
WW:9956710
WZ:4746436
WZQQ:16825067
DY:128601645
TTZ:9833460
ZZ:54791260
H:3750276
HWW:6374523
WWW:3480222
WWZ:8349477



In [11]:
out_dir = DATE+"/Presel/DUMP/"+ERA

In [12]:
os.system("mkdir -p "+out_dir)
with open(out_dir+'/Nevents.txt', 'w') as f:
    f.write(totalinput)

In [13]:
print("expected:", data_evts[ERA], ", while: ", merged_dic_df['Data'][0])

try:
   if data_evts[ERA] > 1.01*merged_dic_df['Data'][0] or data_evts[ERA] < 0.99*merged_dic_df['Data'][0]: # check if more than 1% differ
       raise ValueError('Data events missing!')
except (ValueError, IndexError):
#    beep()
   print("WARNING! expected:", data_evts[ERA], ", while: ", merged_dic_df['Data'][0]) 
   raise

expected: 1326465857 , while:  1326465857


In [14]:
# # temp!!
# for key in merged_dic_df:
#     df = merged_dic_df[key][1]
#     df_met_u = merged_dic_df[key][2]
#     df_met_d = merged_dic_df[key][3]    
#     df_met_ue_u = merged_dic_df[key][4]
#     df_met_ue_d = merged_dic_df[key][5]
#     df_scale_u = merged_dic_df[key][6]
#     df_scale_d = merged_dic_df[key][7]
# #     print(df[df.isin([np.inf, -np.inf]).any(1)])
#     if df.isnull().values.any():
#         print("Warning, nan in",key)
#         i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
#         print(i)
#         df_new = df.drop(i)
#         df_met_u_new = df_met_u.drop(i)
#         df_met_d_new = df_met_d.drop(i)        
#         df_met_ue_u_new = df_met_ue_u
#         df_met_ue_d_new = df_met_ue_d        
#         df_scale_u_new = df_scale_u
#         df_scale_d_new = df_scale_d
# #         print(df_new.isnull().values.any())
#         #workaround for tuple assignment
# #         merged_dic_df[key] = merged_dic_df[key][0], df_new, df_met_u_new, df_met_d_new, df_scale_u, df_scale_d
#         merged_dic_df[key] = merged_dic_df[key][0], df_new, df_met_u_new, df_met_d_new, df_met_ue_u_new, df_met_ue_d_new, df_scale_u_new, df_scale_d_new
        


In [15]:
pd.options.mode.chained_assignment = None
# sys = ["puweight_pre", "l1prefiring_pre", 'effSF_pre', 'topSF_pre', 'genweight_pre', 'trigSF_pre']
sys = ["puweight_pre", "l1prefiring_pre", 'effSF_pre', 'effrecoSF_pre', 'topSF_pre', 'genweight_pre', 'trigSF_pre', 'btagSF_pre'] #, 'btagSF_cor_pre'
# sys = ["puweight_pre", "l1prefiring_pre", 'effSF_pre', 'topSF_pre', 'genweight_pre']
eventweights = """
####################################################
####       event weights (normalization)        ####
####################################################
"""

def get_dic_with_weight(merged_dic_df):
    dic_df={}
    dic_df_met_u={}
    dic_df_met_d={}    
    dic_df_met_jer_u={}
    dic_df_met_jer_d={}
    dic_df_met_ue_u={}
    dic_df_met_ue_d={}
    dic_df_scale_u={}
    dic_df_scale_d={}
    for key, vs in merged_dic_df.items():
        print(key) 
        if vs[0]==0:
            print("missing: ", key, "!, continue without it")
            continue
        if "Data" in key:
            norm=1
        else:
            if fnmatch.fnmatch(key, '*DY*To*'):
                key = key[:len(key)-1]
            norm=lumi*dic_xsec[key]/vs[0]
        print(key, ":", norm)
        global eventweights
        eventweights+=key+":"+str(norm)+"\n"
#         print(eventweights)
        dic_df[key]=vs[1]
        dic_df_met_u[key] = vs[2]
        dic_df_met_d[key] = vs[3]        
        dic_df_met_jer_u[key] = vs[3]
        dic_df_met_jer_d[key] = vs[4]
        dic_df_met_ue_u[key] = vs[5]
        dic_df_met_ue_d[key] = vs[6]
        dic_df_scale_u[key] = vs[7]
        dic_df_scale_d[key]=vs[8]
        dic_df[key]["weight"] = vs[1]["genweight_pre"]*norm
#         dic_df[key]["lumifilter"] = np.ones(vs[1]["genweight_pre"].shape[0])
        a = np.ones(vs[1]["genweight_pre"].shape[0])
        a[int(vs[1]["genweight_pre"].shape[0]*1/3):] = 0
        if key=="Data":
            a = np.zeros(vs[1]["genweight_pre"].shape[0])
#         print(a)
        dic_df_met_u[key]["weightmetUp"] = vs[1]["genweight_pre"]*norm
        dic_df_met_d[key]["weightmetDown"] = vs[1]["genweight_pre"]*norm        
        dic_df_met_jer_u[key]["weightmetjerUp"] = vs[1]["genweight_pre"]*norm
        dic_df_met_jer_d[key]["weightmetjerDown"] = vs[1]["genweight_pre"]*norm
        dic_df_met_ue_u[key]["weightmetueUp"] = vs[1]["genweight_pre"]*norm
        dic_df_met_ue_d[key]["weightmetueDown"] = vs[1]["genweight_pre"]*norm
        dic_df_scale_u[key]["weightscaleUp"] = vs[1]["genweight_pre"]*norm
        dic_df_scale_d[key]["weightscaleDown"] = vs[1]["genweight_pre"]*norm
#         dic_df_met_u[key]["hemveto"] = vs[1]["hemveto"]
#         dic_df_met_d[key]["hemveto"] = vs[1]["hemveto"]       
#         dic_df_met_jer_u[key]["hemveto"] = vs[1]["hemveto"]
#         dic_df_met_jer_d[key]["hemveto"] = vs[1]["hemveto"]
#         dic_df_met_ue_u[key]["hemveto"] = vs[1]["hemveto"]
#         dic_df_met_ue_d[key]["hemveto"] = vs[1]["hemveto"]
#         dic_df_scale_u[key]["hemveto"] = vs[1]["hemveto"]
#         dic_df_scale_d[key]["hemveto"] = vs[1]["hemveto"]
        dic_df[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_met_u[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_met_d[key]["hemfilter"] = a + vs[1]["hemveto"]    
        dic_df_met_jer_u[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_met_jer_d[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_met_ue_u[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_met_ue_d[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_scale_u[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df_scale_d[key]["hemfilter"] = a + vs[1]["hemveto"]
        dic_df[key]["wopu"] = vs[1]["genweight_pre"]*norm
        dic_df[key]["wol1"] = vs[1]["genweight_pre"]*norm
        dic_df[key]["woeff"] = vs[1]["genweight_pre"]*norm
        dic_df[key]["wotop"] = vs[1]["genweight_pre"]*norm       
        dic_df[key]["wobsf"] = vs[1]["genweight_pre"]*norm       
      
        for s in sys:
            if s=="genweight_pre" or s=="btagSF_cor_pre":
                continue
            dic_df[key]["weight"] *= vs[1][s]
            dic_df_met_u[key]["weightmetUp"] *= vs[1][s]
            dic_df_met_d[key]["weightmetDown"] *= vs[1][s]            
            dic_df_met_jer_u[key]["weightmetjerUp"] *= vs[1][s]
            dic_df_met_jer_d[key]["weightmetjerDown"] *= vs[1][s]            
            dic_df_met_ue_u[key]["weightmetueUp"] *= vs[1][s]
            dic_df_met_ue_d[key]["weightmetueDown"] *= vs[1][s]            
            dic_df_scale_u[key]["weightscaleUp"] *= vs[1][s]
            dic_df_scale_d[key]["weightscaleDown"] *= vs[1][s]       

            # n-1 correction
            if s!="puweight_pre":
                dic_df[key]["wopu"] *= vs[1][s]
            if s!="l1prefiring_pre":
                dic_df[key]["wol1"] *= vs[1][s]
            if s!="effSF_pre":
                dic_df[key]["woeff"] *= vs[1][s]
#             if s!="topSF_pre":
            if not "top" in s:
                dic_df[key]["wotop"] *= vs[1][s]
            if s!="btagSF_pre":
                dic_df[key]["wobsf"] *= vs[1][s]

        dic_df[key]["weighthdampUp"] = dic_df[key]["weight"].copy(deep=True)
        dic_df[key]["weighthdampDown"] = dic_df[key]["weight"].copy(deep=True)
        dic_df[key]["weightPSUp"] = dic_df[key]["weight"].copy(deep=True)
        dic_df[key]["weightPSDown"] = dic_df[key]["weight"].copy(deep=True)            
        dic_df[key]["weightJECUp"] = dic_df[key]["weight"].copy(deep=True)/vs[1]["btagSF_pre"]*vs[1]["btagSF_jes_u_pre"]            
        dic_df[key]["weightJECDown"] = dic_df[key]["weight"].copy(deep=True)/vs[1]["btagSF_pre"]*vs[1]["btagSF_jes_d_pre"]         
        dic_df[key]["weightJERUp"] = dic_df[key]["weight"].copy(deep=True)/vs[1]["btagSF_pre"]*vs[1]["btagSF_jer_u_pre"]            
        dic_df[key]["weightJERDown"] = dic_df[key]["weight"].copy(deep=True)/vs[1]["btagSF_pre"]*vs[1]["btagSF_jer_d_pre"]                      
            
        # up, down    
        for s1 in sys:
            #if "top" in s1 or "eff" in s1:
#             if "btag" in s1:
#                 continue
            if s1=="btagSF_pre":
                continue
            s_u = s1.replace("_pre", "_u_pre")
            s_d = s1.replace("_pre", "_d_pre")
            if s1=="genweight_pre":
                dic_df[key]["weight"+s_u.replace('_u_pre', 'Up')] = dic_df[key]["weight"].copy(deep=True)*np.fabs(dic_df[key][s_u])
                dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)*np.fabs(dic_df[key][s_d])
#             elif s1=="btagSF_cor_pre":   
#                 dic_df[key]["weight"+s_u.replace('_u_pre', 'Up')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key]['btagSF_pre']*dic_df[key][s_u]
#                 dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key]['btagSF_pre']*dic_df[key][s_d] 
            else:
                dic_df[key]["weight"+s_u.replace('_u_pre', 'Up')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key][s1]*dic_df[key][s_u]
                dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key][s1]*dic_df[key][s_d]  
                
        # for b-tag
        btag_sys = ['btagSFbc_pre', 'btagSFbc_correlated_pre', 'btagSFlight_pre', 'btagSFlight_correlated_pre']
        for bsys in btag_sys:
            s_u = bsys.replace("_pre", "_u_pre")
            s_d = bsys.replace("_pre", "_d_pre")
            dic_df[key]["weight"+s_u.replace('_u_pre', 'Up')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key]['btagSF_pre']*dic_df[key][s_u]
            dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key]['btagSF_pre']*dic_df[key][s_d]
        
                
#     return dic_df, dic_df_met_u, dic_df_met_d, dic_df_scale_u, dic_df_scale_d
    return dic_df, dic_df_met_u, dic_df_met_d, dic_df_met_jer_u, dic_df_met_jer_d, dic_df_met_ue_u, dic_df_met_ue_d, dic_df_scale_u, dic_df_scale_d

# dic_df, dic_df_met_u, dic_df_met_d, dic_df_scale_u, dic_df_scale_d = get_dic_with_weight(merged_dic_df)
dic_df, dic_df_met_u, dic_df_met_d, dic_df_met_jer_u, dic_df_met_jer_d, dic_df_met_ue_u, dic_df_met_ue_d, dic_df_scale_u, dic_df_scale_d = get_dic_with_weight(merged_dic_df)


DY1000To15000
DY1000To1500 : 0.009785467982499308
DY1500To20000
DY1500To2000 : 0.0012113801686825314
DY2000To30000
DY2000To3000 : 0.00028492232043809756
DY500To7000
DY500To700 : 0.09659806229684098
DY700To8000
DY700To800 : 0.021584576686126282
DY800To10000
DY800To1000 : 0.01815582501781896
DY
DY : 2.8273360935624114
Data
Data : 1
HWW
HWW : 0.002909598098555766
H
H : 0.044350709121142015
TTDown
TTDown : 0.08897495397124701
TTSemi
TTSemi : 0.046255294195546054
TTUp
TTUp : 0.09332291098458388
TTZ
TTZ : 0.001479708668159529
TT
TT : 0.03672181441453045
TThdampDown
TThdampDown : 0.08957928593054988
TThdampUp
TThdampUp : 0.09817944499744514
WWW
WWW : 0.00012377831069397297
WWZ
WWZ : 0.001223188111063723
WW
WW : 0.07703554688245415
WZQQ
WZQQ : 0.02282954356140157
WZ
WZ : 0.06592544384881625
ZZ
ZZ : 0.0006560510563181063
Zp-1700_CH-345
Zp-1700_CH-345 : 0.014052072666666667
Zp-1700_CH-595
Zp-1700_CH-595 : 0.014052072666666667
Zp-1700_CH-845
Zp-1700_CH-845 : 0.014052072666666667
Zp-2100_CH-345
Zp

In [16]:
pd.set_option('display.max_columns', None)
dic_df["TT"][:10]

,PuppiMET_pre,mass_pre,pt_lead_pre,pt_trail_pre,U_pre,JZB_pre,dPhill_pre,dPhilMET_pre,MT_pre,MT2_pre,Zpt_pre,dRll_pre,dPhillMET_pre,eta_lead_pre,eta_trail_pre,njet_pre,nbjetflav_pre,nbjetflav_jesup_pre,nbjetflav_jesdown_pre,nbjetflav_jerup_pre,nbjetflav_jerdown_pre,HT_pre,PuppiMET_phi_pre,MET_pre,MET_phi_pre,phi_lead_pre,phi_trail_pre,hemveto,genweight_u_pre,genweight_d_pre,genweight_pre,puweight_pre,puweight_u_pre,puweight_d_pre,l1prefiring_pre,l1prefiring_u_pre,l1prefiring_d_pre,effrecoSF_pre,effrecoSF_u_pre,effrecoSF_d_pre,effSF_pre,effSF_u_pre,effSF_d_pre,trigSF_pre,trigSF_u_pre,trigSF_d_pre,topSF_pre,topSF_u_pre,topSF_d_pre,btagSF_pre,btagSFbc_u_pre,btagSFbc_d_pre,btagSFbc_correlated_u_pre,btagSFbc_correlated_d_pre,btagSFlight_u_pre,btagSFlight_d_pre,btagSFlight_correlated_u_pre,btagSFlight_correlated_d_pre,btagSF_jes_u_pre,btagSF_jes_d_pre,btagSF_jer_u_pre,btagSF_jer_d_pre,pdf0_pre,pdf1_pre,pdf2_pre,pdf3_pre,pdf4_pre,pdf5_pre,pdf6_pre,pdf7_pre,pdf8_pre,pdf9_pre,pdf10_pre,pdf11_pre,pdf12_pre,pdf13_pre,pdf14_pre,pdf15_pre,pdf16_pre,pdf17_pre,pdf18_pre,pdf19_pre,pdf20_pre,pdf21_pre,pdf22_pre,pdf23_pre,pdf24_pre,pdf25_pre,pdf26_pre,pdf27_pre,pdf28_pre,pdf29_pre,pdf30_pre,pdf31_pre,pdf32_pre,pdf33_pre,pdf34_pre,pdf35_pre,pdf36_pre,pdf37_pre,pdf38_pre,pdf39_pre,pdf40_pre,pdf41_pre,pdf42_pre,pdf43_pre,pdf44_pre,pdf45_pre,pdf46_pre,pdf47_pre,pdf48_pre,pdf49_pre,pdf50_pre,pdf51_pre,pdf52_pre,pdf53_pre,pdf54_pre,pdf55_pre,pdf56_pre,pdf57_pre,pdf58_pre,pdf59_pre,pdf60_pre,pdf61_pre,pdf62_pre,pdf63_pre,pdf64_pre,pdf65_pre,pdf66_pre,pdf67_pre,pdf68_pre,pdf69_pre,pdf70_pre,pdf71_pre,pdf72_pre,pdf73_pre,pdf74_pre,pdf75_pre,pdf76_pre,pdf77_pre,pdf78_pre,pdf79_pre,pdf80_pre,pdf81_pre,pdf82_pre,pdf83_pre,pdf84_pre,pdf85_pre,pdf86_pre,pdf87_pre,pdf88_pre,pdf89_pre,pdf90_pre,pdf91_pre,pdf92_pre,pdf93_pre,pdf94_pre,pdf95_pre,pdf96_pre,pdf97_pre,pdf98_pre,pdf99_pre,pdf100_pre,psweight1_pre,psweight2_pre,psweight3_pre,psweight4_pre,scale1_pre,scale2_pre,scale3_pre,scale4_pre,scale5_pre,scale6_pre,weight,hemfilter,wopu,wol1,woeff,wotop,wobsf,weighthdampUp,weighthdampDown,weightPSUp,weightPSDown,weightJECUp,weightJECDown,weightJERUp,weightJERDown,weightpuweightUp,weightpuweightDown,weightl1prefiringUp,weightl1prefiringDown,weighteffSFUp,weighteffSFDown,weighteffrecoSFUp,weighteffrecoSFDown,weighttopSFUp,weighttopSFDown,weightgenweightUp,weightgenweightDown,weighttrigSFUp,weighttrigSFDown,weightbtagSFbcUp,weightbtagSFbcDown,weightbtagSFbc_correlatedUp,weightbtagSFbc_correlatedDown,weightbtagSFlightUp,weightbtagSFlightDown,weightbtagSFlight_correlatedUp,weightbtagSFlight_correlatedDown
0,106.010635,132.896545,132.609207,39.587227,8.911835,-100.587593,2.321515,2.950195,227.194046,54.067062,109.499428,2.321516,3.065461,1.186005,1.187042,1,0,0,0,0,0,34.661224,1.394287,116.263626,1.663818,-1.555908,2.405762,1.0,1.0,1.0,1.0,0.882101,0.710251,1.097186,1.000000,1.000000,1.000000,0.973058,0.983629,0.962534,0.968165,0.983213,0.953248,0.994591,0.996522,0.992599,0.965307,1.0,0.965307,0.996754,0.996754,0.996754,0.996754,0.996754,0.996131,0.997376,0.995250,0.998258,0.996754,0.996754,0.996754,0.996754,0.999969,0.999359,1.000793,1.000061,1.002380,0.999786,0.998138,0.999420,0.998322,1.001221,0.999542,0.997223,0.998810,0.997284,0.999298,0.999512,1.000977,0.999390,0.998535,1.000305,0.998657,1.000854,0.998199,1.000549,0.997864,0.998535,1.001099,0.998657,0.998840,1.006104,1.000244,1.000366,1.000549,1.000549,1.001953,0.999390,0.999146,1.001770,0.999329,0.998047,0.999146,0.998901,0.999084,0.999603,1.001160,0.999390,1.002075,1.001526,0.998993,0.999481,1.001221,1.000732,0.999359,1.000244,1.000610,1.000366,0.999847,0.999664,1.000244,0.999817,1.000305,0.999756,1.000061,0.999908,0.999695,1.000244,0.999817,1.000183,1.000244,1.000244,0.999451,0.999298,1.000000,0.999725,0.999542,0.999908,1.000061,0.999512,1.000183,1.000183,1.000061,1.000244,1.000244,0.999939,1.000000,0.999969,0.999939,0.999817,1.000244,0.999939,1.000244,0.999939,0.999969,0.999969,0.999939,1.000061,0.999817,1.0

In [17]:
with open(out_dir+'/Nevents.txt', 'a') as f:
    f.write(eventweights)

# Preselection

In [18]:
dic_df_list = [dic_df, dic_df_met_u, dic_df_met_d, dic_df_met_jer_u, dic_df_met_jer_d, dic_df_met_ue_u, dic_df_met_ue_d, dic_df_scale_u, dic_df_scale_d]
dic_key_list = ['raw', 'met_u', 'met_d', 'met_jer_u', 'met_jer_d', 'met_ue_u', 'met_ue_d', 'scale_u', 'scale_d']

In [19]:
def get_nested_dic(dic_df_list, dic_key_list, query_str):
    weihtedevents = """
    #############################################################
    ####   unweighted (weighted) #events after preselection  ####
    #############################################################
    """

    nested_dic_df={}

    for i, dic in enumerate(dic_df_list):
        dic_tmp = {}
        for key in dic:
#             query_str_new = query_str
#             if "18" in ERA and key=='Data':
#                 query_str_new += " and hemveto==1"
#             if "18" in ERA and key!='Data':
#                 query_str_new += " and hemfilter>0"
            dic_tmp[key]=dic[key].query(query_str)
            unweighted_nevt = dic_tmp[key].shape[0]
            if dic_key_list[i]=='raw':
                print(key, dic_key_list[i], ":", unweighted_nevt, "(", sum(dic_tmp[key]["weight"]), ")")  
                weihtedevents+=key+":"+str(unweighted_nevt)+"("+str(sum(dic_tmp[key]["weight"]))+")\n"
            else:
                print(key, dic_key_list[i], ":", dic_tmp[key].shape[0])
        #     print(key, ":", unweighted_nevt, "(", unweighted_nevt*lumi*dic_xsec[key]/merged_dic_df[key][0], ")")
        nested_dic_df[dic_key_list[i]]=dic_tmp
    
    return nested_dic_df, weihtedevents

In [20]:
# query_str="mass_pre>100 and pt_lead_pre>80 and pt_trail_pre>40 and PuppiMET_pre>100"

# nested_dic_df, weihtedevents = get_nested_dic(dic_df_list, dic_key_list, query_str)

In [21]:
# with open(out_dir+'/Nevents.txt', 'a') as f:
#     f.write(weihtedevents)

In [22]:
# if DUMP_PKL or "16" in ERA:
#     print("Dump pickle")
#     with open(out_dir+"/"+"nested_dic_df.pkl", "wb") as f:
#         pickle.dump(nested_dic_df, f)

# b tag selection

In [23]:
if DUMP_DIVIDED_BY_BTAG:

    base_line = "mass_pre>100 and pt_lead_pre>80 and pt_trail_pre>40 and PuppiMET_pre>100"
    
    if "18" in ERA:
        base_line += " and hemfilter>0"
#     query_str="mass_pre>100 and pt_lead_pre>80 and pt_trail_pre>40 and PuppiMET_pre>100 and njet_pre==1" #temp!
    query_str=base_line+" and nbjetflav_pre==0" #temp!
#     query_str=base_line #temp!

    nested_dic_df, weihtedevents = get_nested_dic(dic_df_list, dic_key_list, query_str)
    # nbjet_jec
    dic_tmp = {}
    selected_columns = ['PuppiMET_pre', 'PuppiMET_phi_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'eta_lead_pre', 'eta_trail_pre',
       'njet_pre', 'nbjetflav_pre', 'nbjetflav_jesup_pre', 'weightJECUp', 'hemfilter']    
    for key in dic_df:  
        dic_tmp[key]=dic_df[key][selected_columns].query(base_line+" and nbjetflav_jesup_pre==0")
        unweighted_nevt = dic_tmp[key].shape[0]
    nested_dic_df['jes_u'] = dic_tmp
    # nbjet_jec
    dic_tmp = {}
    selected_columns = ['PuppiMET_pre', 'PuppiMET_phi_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'eta_lead_pre', 'eta_trail_pre',
       'njet_pre', 'nbjetflav_pre', 'nbjetflav_jesdown_pre', 'weightJECDown', 'hemfilter']    
    for key in dic_df:  
        dic_tmp[key]=dic_df[key][selected_columns].query(base_line+" and nbjetflav_jesdown_pre==0")
        unweighted_nevt = dic_tmp[key].shape[0]    
    nested_dic_df['jes_d'] = dic_tmp
    # nbjet_jer
    dic_tmp = {}
    selected_columns = ['PuppiMET_pre', 'PuppiMET_phi_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'eta_lead_pre', 'eta_trail_pre',
       'njet_pre', 'nbjetflav_pre', 'nbjetflav_jerup_pre', 'weightJERUp', 'hemfilter']    
    for key in dic_df:  
        dic_tmp[key]=dic_df[key][selected_columns].query(base_line+" and nbjetflav_jerup_pre==0")
        unweighted_nevt = dic_tmp[key].shape[0]
    nested_dic_df['jer_u'] = dic_tmp
    # nbjet_jer
    dic_tmp = {}
    selected_columns = ['PuppiMET_pre', 'PuppiMET_phi_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'eta_lead_pre', 'eta_trail_pre',
       'njet_pre', 'nbjetflav_pre', 'nbjetflav_jerdown_pre', 'weightJERDown', 'hemfilter']    
    for key in dic_df:  
        dic_tmp[key]=dic_df[key][selected_columns].query(base_line+" and nbjetflav_jerdown_pre==0")
        unweighted_nevt = dic_tmp[key].shape[0]    
    nested_dic_df['jer_d'] = dic_tmp    
    with open(out_dir+"/"+"nested_dic_df_SR1.pkl", "wb") as f:
        pickle.dump(nested_dic_df, f)     
    with open(out_dir+"/"+"nested_dic_df_SR1_raw.pkl", "wb") as f:
        pickle.dump(nested_dic_df['raw'], f) 
        
    nested_dic_df.clear()
    
    ####################################################
    query_str=base_line+" and nbjetflav_pre>0" #temp!

    nested_dic_df, weihtedevents = get_nested_dic(dic_df_list, dic_key_list, query_str)
    
    # nbjet_jec
    dic_tmp = {}
    selected_columns = ['PuppiMET_pre', 'PuppiMET_phi_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'eta_lead_pre', 'eta_trail_pre',
       'njet_pre', 'nbjetflav_pre', 'nbjetflav_jesup_pre', 'weightJECUp', 'hemfilter']    
    for key in dic_df:  
        dic_tmp[key]=dic_df[key][selected_columns].query(base_line+" and nbjetflav_jesup_pre>0")
        unweighted_nevt = dic_tmp[key].shape[0]
    nested_dic_df['jes_u'] = dic_tmp
    # nbjet_jec
    dic_tmp = {}
    selected_columns = ['PuppiMET_pre', 'PuppiMET_phi_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'eta_lead_pre', 'eta_trail_pre',
       'njet_pre', 'nbjetflav_pre', 'nbjetflav_jesdown_pre', 'weightJECDown', 'hemfilter']    
    for key in dic_df:  
        dic_tmp[key]=dic_df[key][selected_columns].query(base_line+" and nbjetflav_jesdown_pre>0")
        unweighted_nevt = dic_tmp[key].shape[0]    
    nested_dic_df['jes_d'] = dic_tmp
    # nbjet_jer
    dic_tmp = {}
    selected_columns = ['PuppiMET_pre', 'PuppiMET_phi_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'eta_lead_pre', 'eta_trail_pre',
       'njet_pre', 'nbjetflav_pre', 'nbjetflav_jerup_pre', 'weightJERUp', 'hemfilter']    
    for key in dic_df:  
        dic_tmp[key]=dic_df[key][selected_columns].query(base_line+" and nbjetflav_jerup_pre>0")
        unweighted_nevt = dic_tmp[key].shape[0]
    nested_dic_df['jer_u'] = dic_tmp
    # nbjet_jer
    dic_tmp = {}
    selected_columns = ['PuppiMET_pre', 'PuppiMET_phi_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'eta_lead_pre', 'eta_trail_pre',
       'njet_pre', 'nbjetflav_pre', 'nbjetflav_jerdown_pre', 'weightJERDown', 'hemfilter']    
    for key in dic_df:  
        dic_tmp[key]=dic_df[key][selected_columns].query(base_line+" and nbjetflav_jerdown_pre>0")
        unweighted_nevt = dic_tmp[key].shape[0]    
    nested_dic_df['jer_d'] = dic_tmp 
    
    with open(out_dir+"/"+"nested_dic_df_TTCR.pkl", "wb") as f:
        pickle.dump(nested_dic_df, f) 
    with open(out_dir+"/"+"nested_dic_df_TTCR_raw.pkl", "wb") as f:
        pickle.dump(nested_dic_df['raw'], f)         
    nested_dic_df.clear()    

DY1000To1500 raw : 885 ( 4.5544263161718845 )
DY1500To2000 raw : 1355 ( 0.8902345080859959 )
DY2000To3000 raw : 1767 ( 0.27665199733746704 )
DY500To700 raw : 376 ( 17.025757387280464 )
DY700To800 raw : 460 ( 5.553893743082881 )
DY800To1000 raw : 658 ( 6.016509698703885 )
DY raw : 238 ( 332.69022035598755 )
Data raw : 3832 ( 3832.0 )
HWW raw : 0 ( 0 )
H raw : 1 ( 0.04512173309922218 )
TTDown raw : 27702 ( 2398.269113138318 )
TTSemi raw : 118 ( 5.387907080352306 )
TTUp raw : 26337 ( 2385.6842609345913 )
TTZ raw : 4342 ( 3.3469770298106596 )
TT raw : 67256 ( 2397.973140483722 )
TThdampDown raw : 27478 ( 2363.121673192829 )
TThdampUp raw : 25533 ( 2448.35399607569 )
WWW raw : 22438 ( 2.397409820667235 )
WWZ raw : 6280 ( 6.6263620820245706 )
WW raw : 7593 ( 555.0663173347712 )
WZQQ raw : 233 ( 2.982912650331855 )
WZ raw : 2944 ( 96.15225483104587 )
ZZ raw : 31783 ( 19.81240292536677 )
Zp-1700_CH-345 raw : 9506 ( 128.92904011253268 )
Zp-1700_CH-595 raw : 11530 ( 156.06842839997262 )
Zp-1700_

Zp-2900_CH-345 met_jer_d : 13342
Zp-2900_CH-595 met_jer_d : 14905
Zp-2900_CH-845 met_jer_d : 16175
Zp-3300_CH-1095 met_jer_d : 17581
Zp-3300_CH-1345 met_jer_d : 18198
Zp-3300_CH-1595 met_jer_d : 18566
Zp-3300_CH-345 met_jer_d : 14149
Zp-3300_CH-595 met_jer_d : 15517
Zp-3300_CH-845 met_jer_d : 16752
Zp-3700_CH-1095 met_jer_d : 17979
Zp-3700_CH-1345 met_jer_d : 18504
Zp-3700_CH-1595 met_jer_d : 18780
Zp-3700_CH-1845 met_jer_d : 19327
Zp-3700_CH-345 met_jer_d : 14754
Zp-3700_CH-595 met_jer_d : 16042
Zp-3700_CH-845 met_jer_d : 17079
Zp-4100_CH-1095 met_jer_d : 17261
Zp-4100_CH-1345 met_jer_d : 18774
Zp-4100_CH-1595 met_jer_d : 19058
Zp-4100_CH-1845 met_jer_d : 19498
Zp-4100_CH-345 met_jer_d : 15269
Zp-4100_CH-595 met_jer_d : 16385
Zp-4100_CH-845 met_jer_d : 17154
antitop met_jer_d : 744
top met_jer_d : 660
DY1000To1500 met_ue_u : 791
DY1500To2000 met_ue_u : 1231
DY2000To3000 met_ue_u : 1646
DY500To700 met_ue_u : 368
DY700To800 met_ue_u : 420
DY800To1000 met_ue_u : 600
DY met_ue_u : 219
Dat

Zp-2100_CH-845 raw : 429 ( 2.1104990877211094 )
Zp-2500_CH-345 raw : 424 ( 0.7822578387567773 )
Zp-2500_CH-595 raw : 480 ( 0.8646620691288263 )
Zp-2500_CH-845 raw : 484 ( 0.8658829320920631 )
Zp-2900_CH-1095 raw : 602 ( 0.4368593090912327 )
Zp-2900_CH-1345 raw : 587 ( 0.42202327627455816 )
Zp-2900_CH-345 raw : 492 ( 0.35989252268336713 )
Zp-2900_CH-595 raw : 514 ( 0.373917642980814 )
Zp-2900_CH-845 raw : 520 ( 0.3751271333312616 )
Zp-3300_CH-1095 raw : 560 ( 0.16824057669145986 )
Zp-3300_CH-1345 raw : 585 ( 0.1753076269669691 )
Zp-3300_CH-1595 raw : 668 ( 0.199379791985848 )
Zp-3300_CH-345 raw : 534 ( 0.16040851961588487 )
Zp-3300_CH-595 raw : 561 ( 0.16759324006852694 )
Zp-3300_CH-845 raw : 574 ( 0.170211698350613 )
Zp-3700_CH-1095 raw : 632 ( 0.08012836300622439 )
Zp-3700_CH-1345 raw : 671 ( 0.08532202157221036 )
Zp-3700_CH-1595 raw : 701 ( 0.08962497804895975 )
Zp-3700_CH-1845 raw : 666 ( 0.08489423453283962 )
Zp-3700_CH-345 raw : 545 ( 0.06894542592635844 )
Zp-3700_CH-595 raw : 606

Zp-3300_CH-345 met_ue_u : 537
Zp-3300_CH-595 met_ue_u : 559
Zp-3300_CH-845 met_ue_u : 576
Zp-3700_CH-1095 met_ue_u : 634
Zp-3700_CH-1345 met_ue_u : 667
Zp-3700_CH-1595 met_ue_u : 703
Zp-3700_CH-1845 met_ue_u : 667
Zp-3700_CH-345 met_ue_u : 540
Zp-3700_CH-595 met_ue_u : 607
Zp-3700_CH-845 met_ue_u : 637
Zp-4100_CH-1095 met_ue_u : 604
Zp-4100_CH-1345 met_ue_u : 665
Zp-4100_CH-1595 met_ue_u : 729
Zp-4100_CH-1845 met_ue_u : 708
Zp-4100_CH-345 met_ue_u : 575
Zp-4100_CH-595 met_ue_u : 600
Zp-4100_CH-845 met_ue_u : 644
antitop met_ue_u : 1377
top met_ue_u : 1277
DY1000To1500 met_ue_d : 26
DY1500To2000 met_ue_d : 74
DY2000To3000 met_ue_d : 76
DY500To700 met_ue_d : 26
DY700To800 met_ue_d : 30
DY800To1000 met_ue_d : 46
DY met_ue_d : 32
Data met_ue_d : 8485
HWW met_ue_d : 0
H met_ue_d : 1
TTDown met_ue_d : 104843
TTSemi met_ue_d : 331
TTUp met_ue_d : 100486
TTZ met_ue_d : 17501
TT met_ue_d : 253844
TThdampDown met_ue_d : 104031
TThdampUp met_ue_d : 96071
WWW met_ue_d : 1722
WWZ met_ue_d : 1100
WW

In [24]:
sys_list = ["scale", "puweight", "l1prefiring", "effrecoSF", "effSF", "topSF", "met", "metjer", "metue", "trigSF", "hdamp", "PS", "btagSF", "jes", "jer"]
btag_sys = ['btagSFbc', 'btagSFbc_correlated', 'btagSFlight', 'btagSFlight_correlated']
sys_list_new = sys_list + btag_sys
sys_list_new.remove('btagSF')
print(sys_list_new)

['scale', 'puweight', 'l1prefiring', 'effrecoSF', 'effSF', 'topSF', 'met', 'metjer', 'metue', 'trigSF', 'hdamp', 'PS', 'jes', 'jer', 'btagSFbc', 'btagSFbc_correlated', 'btagSFlight', 'btagSFlight_correlated']
